# Swar — Nepali / English voice-cloning TTS on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/swar-voice-clone-tts-ffa3/colab/Swar-voice-clone-tts.ipynb)

[`Aarjan/swar`](https://huggingface.co/Aarjan/swar) is **Swar** (स्वर, "voice") from [Naamche Labs](https://naamchelabs.com/models/swar): an open Nepali voice-cloning text-to-speech model. It reads **Nepali, English, and Nepali mixed with English**, and clones a voice from about 20–30 seconds of one speaker. The architecture is **F5-TTS Base**, initialised from [AI4Bharat IndicF5](https://huggingface.co/ai4bharat/IndicF5) (MIT) and fine-tuned on open Nepali and English speech plus podcast recordings used with the speakers' permission.

This is the free-Colab Nepali/English cloning notebook. The [Phonon 31M notebook](phonon-31m-en-streaming-tts.ipynb) is tiny English-only CPU TTS. The [Maya 2 Native notebook](Maya2-Native-indic-tts.ipynb) is 11 Indic languages + English with built-in voices. The [Drama 3 notebook](Drama-3-tts-voice-clone.ipynb) needs a Colab Pro A100.

The Hugging Face repo is **weights + presets + the Gradio cloner**. Inference is the official PyPI package [`f5-tts`](https://pypi.org/project/f5-tts/) (`f5_tts.api.F5TTS`) plus [Naamche-Labs/swar](https://github.com/Naamche-Labs/swar) text prep. The authors also ship [`notebooks/Swar_Clone_Colab.ipynb`](https://github.com/Naamche-Labs/swar/blob/main/notebooks/Swar_Clone_Colab.ipynb), which installs the same `f5-tts==1.1.22` pin and launches Gradio. This notebook follows that install/load path, then calls `F5TTS.infer` the way their `pipeline/autoclone.py` does — it does **not** vendor their Gradio app.

**Licence (read this):** Weights (`podvoices_ema.pt`, `swar_v6_ema.pt`, `presets/`) are **[CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)** — share-alike, **commercial use is allowed**, attribution required. Downstream weights you train from Swar must stay CC BY-SA 4.0. Code is [MIT](https://github.com/Naamche-Labs/swar/blob/main/LICENSE). Kriti (the optional ONNX CTC recogniser) is MIT. This is **not** the lookalike Hub repo `elitexp/Swar` (Kokoro-family, CC BY-NC). It is also **not** the default `SWivid/F5-TTS` English checkpoint (CC BY-NC). This notebook only loads `Aarjan/swar` / `podvoices_ema.pt`.

**Consent:** Only clone a voice with that person's permission. Do not impersonate anyone, deceive listeners, or commit fraud. The Maya and Aarav presets are **model-made audio**, not recordings of real people. Label synthetic speech as synthetic where it could be mistaken for a person.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~15–16 GB of VRAM and no native bfloat16. Default weights are `podvoices_ema.pt` (~**1.35 GB**). The full cloner also ships `kriti_onnx/kriti_ctc.onnx` (~**0.49 GB** ONNX CTC) to transcribe uploads; this notebook does **not** download or load it. A second checkpoint `swar_v6_ema.pt` (~1.35 GB) is better on English-heavy text and weaker on pure Nepali — it is also skipped. IndicF5-derived weights produce **NaN / noise in float16**; `autoclone.py` upcasts `ema_model` to **float32**. Attention stays whatever `f5-tts` uses (no `flash-attn`). Peak VRAM for a short clip at NFE 16 should stay well under 8 GB.

**Not in this notebook:** the Gradio Space / official cloner UI, Kriti enrolment, multi-take scoring, `swar_v6_ema.pt` / `ENGLISH_LATIN=1`, BigVGAN, Chatterbox, and cloning a real person's voice.

**Links:** [model card](https://huggingface.co/Aarjan/swar) · [GitHub](https://github.com/Naamche-Labs/swar) · [official Colab](https://colab.research.google.com/github/Naamche-Labs/swar/blob/main/notebooks/Swar_Clone_Colab.ipynb) · [cloning bench](https://github.com/Naamche-Labs/nepali-voice-cloning-bench) · [announcement](https://x.com/ArjanChaudharyy/status/2115809203218755809)


## 1. Install official `f5-tts` and the Swar frontend

The authors' Colab pin is `f5-tts==1.1.22` from **PyPI** ([SWivid/F5-TTS](https://github.com/SWivid/F5-TTS)). Install **that package name only**. Do not install lookalikes (`f5tts`, `f5_tts`, `openf5-tts`, `F5-TTS-fork`, …). Do not replace Colab's CUDA torch.

The Naamche Labs repo is cloned for Nepali text normalisation (`frontend/`) and the F5 chunker / pace helpers (`pipeline/clone.py`). Weights come from the Hub in the next cells, not from this clone.


In [ ]:
import importlib.metadata
import importlib.util
import os
import subprocess
import sys

print("python", sys.version.split()[0])

try:
    import torch

    print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
except ImportError:
    print("torch not imported yet")


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


# Official Colab: pip install "f5-tts==1.1.22" …
# Do not add speechbrain / onnxruntime-gpu — those are for the Gradio cloner + Kriti.
# Do not pin torch / torchaudio.
pip_install("f5-tts==1.1.22", "librosa", "soundfile", "scipy", "huggingface_hub")

# Reject a lookalike import name if someone later edits the install line.
assert importlib.util.find_spec("f5_tts") is not None, (
    "Official package is PyPI 'f5-tts' (import f5_tts). "
    "Do not install a lookalike name."
)

REPO = "/content/swar" if os.path.isdir("/content") else os.path.abspath("swar")
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "https://github.com/Naamche-Labs/swar.git", REPO]
    )
print("swar", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
print("REPO", REPO)
print("f5-tts", importlib.metadata.version("f5-tts"))


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
>
> Colab has already imported its preinstalled `transformers` (and often `torch`). The `f5-tts` install is invisible until the process restarts. The git clone on disk survives. You do **not** need to rerun the install cell after the restart unless it failed.


## 2. GPU check

Need a CUDA GPU. Print the device name, compute capability, and free VRAM. F5-TTS Base + float32 EMA fits a 15 GB T4; this notebook still refuses a CPU runtime because a short clip is slow without CUDA.


In [ ]:
import os
import shutil

import torch

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the GPU-check cell.")

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

free_b, total_b = torch.cuda.mem_get_info()
major, minor = torch.cuda.get_device_capability()
props = torch.cuda.get_device_properties(0)

REPO = "/content/swar" if os.path.isdir("/content/swar") else os.path.abspath("swar")
MODEL_ID = "Aarjan/swar"
CKPT_NAME = "podvoices_ema.pt"
NFE_STEP = 16  # Swar bench / autoclone default
CFG_STRENGTH = 2.0
SEED = 0

print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}")
print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
print(f"device total: {props.total_memory / 1024**3:.1f} GB")
if major < 8:
    print("No native bfloat16 on this GPU; Swar inference is float32 EMA anyway.")
print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")
print("REPO", REPO)


## 3. Load `podvoices_ema.pt` on the T4

Same constructor as `pipeline/autoclone.py`: `F5TTS(model="F5TTS_Base", ckpt_file=…, vocab_file=…)`, then `ema_model.float()`. `F5TTS_Base` is the IndicF5 / original F5-TTS Base config — **not** `F5TTS_v1_Base` (that would pull the English CC-BY-NC default).

The first run downloads about **1.35 GB** (`podvoices_ema.pt`) plus `vocab.txt`, the Maya/Aarav presets, and the Vocos vocoder `f5-tts` fetches on load. `swar_v6_ema.pt` and `kriti_onnx/` are excluded on purpose.


In [ ]:
import json
import os
import sys

import torch
from huggingface_hub import snapshot_download
from IPython.display import Audio, display

if REPO not in sys.path:
    sys.path.insert(0, REPO)
if os.path.join(REPO, "pipeline") not in sys.path:
    sys.path.insert(0, os.path.join(REPO, "pipeline"))

from f5_tts.api import F5TTS

if MODEL_ID != "Aarjan/swar":
    raise SystemExit("This notebook only loads Aarjan/swar (CC BY-SA 4.0).")

SWAR_DIR = snapshot_download(
    MODEL_ID,
    allow_patterns=["podvoices_ema.pt", "vocab.txt", "presets/**", "README.md"],
)
CKPT = os.path.join(SWAR_DIR, CKPT_NAME)
VOCAB = os.path.join(SWAR_DIR, "vocab.txt")
if os.path.basename(CKPT) != "podvoices_ema.pt":
    raise SystemExit("This notebook loads the default podvoices_ema.pt checkpoint only.")
if not os.path.isfile(CKPT) or not os.path.isfile(VOCAB):
    raise SystemExit(f"missing {CKPT_NAME} or vocab.txt under {SWAR_DIR}")

print("weights", SWAR_DIR)
print(f"{CKPT_NAME}: {os.path.getsize(CKPT) / 1024**3:.2f} GB")

# Authors' frontend: Nepal pronunciation + F5 chunker that splits on Devanagari danda.
from clone import patch_f5_chunker, set_pace, speed_for, to_model_text  # noqa: E402
from frontend import normalize  # noqa: E402

patch_f5_chunker()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

tts = F5TTS(model="F5TTS_Base", ckpt_file=CKPT, vocab_file=VOCAB, device="cuda")
# autoclone.py / clone.py: IndicF5-derived weights give NaN or noise in fp16.
tts.ema_model.float()

voice = json.load(open(os.path.join(SWAR_DIR, "presets/maya/voice.json"), encoding="utf-8"))
prompt = voice["prompts"][0]
REF_WAV = os.path.join(SWAR_DIR, "presets/maya/pieces", os.path.basename(prompt["wav"]))
REF_TEXT = prompt["text"]
if not os.path.isfile(REF_WAV):
    raise SystemExit(f"Maya preset piece missing: {REF_WAV}")

print("model", MODEL_ID, "| arch F5TTS_Base | ckpt", CKPT_NAME)
print("device", tts.device, "| sample_rate", tts.target_sample_rate, "| vocoder", tts.mel_spec_type)
print("ema dtype", next(tts.ema_model.parameters()).dtype)
print("Maya prompt", os.path.basename(REF_WAV), f"({os.path.getsize(REF_WAV) / 1024:.0f} KB)")
print("Maya transcript:", REF_TEXT)
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB | "
    f"reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

RUNS = []
print("Maya preset (model-made, CC BY-SA 4.0 — not a real person's recording):")
display(Audio(REF_WAV))


## 4. Demo 1 — Nepali and English with the built-in Maya voice

F5-TTS always conditions on a reference clip plus its transcript. There is no “no-ref” `speak(text)` API. The built-in path is the **Maya** preset (female, read style, *model-made audio only*). No user upload. `nfe_step=16` and `cfg_strength=2.0` match the published Swar bench.

Text is passed through `to_model_text(..., "chars")`, the same frontend as the cloner: Nepali stays Devanagari, English is written as Nepali-letter English (the default `podvoices_ema.pt` path). `swar_v6_ema.pt` would keep Latin letters; that checkpoint is not loaded here.


In [ ]:
import time

import numpy as np
import soundfile as sf

NE_TEXT = "नमस्ते! आजको मौसम राम्रो छ।"
MIX_TEXT = "नमस्ते! आज office मा meeting धेरै लामो भयो।"
EN_TEXT = "Hello from Swar. Your package will arrive tomorrow morning."


def speak(title, text, *, ref_wav, ref_text, wav_path):
    gen = to_model_text(text, "chars")
    rt = normalize(ref_text)
    set_pace(rt, ref_wav)
    speed = speed_for(gen, 1.0)

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    wav, sr, _spec = tts.infer(
        ref_file=ref_wav,
        ref_text=rt,
        gen_text=gen,
        nfe_step=NFE_STEP,
        cfg_strength=CFG_STRENGTH,
        speed=speed,
        seed=SEED,
        remove_silence=True,
        file_wave=wav_path,
    )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    wave = np.asarray(wav, dtype=np.float32).reshape(-1)
    finite = bool(np.isfinite(wave).all()) and wave.size > 0
    if not finite:
        raise RuntimeError(
            f"{title}: empty or non-finite waveform. IndicF5-derived weights need ema_model.float()."
        )
    audio_s = wave.size / sr
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(
        f"{title}: {audio_s:.2f}s audio in {elapsed:.1f}s "
        f"(RTF {elapsed / max(audio_s, 1e-6):.2f}) | "
        f"VRAM {allocated:.2f} GB allocated, {peak:.2f} GB peak | wrote {wav_path}"
    )
    display(Audio(wave, rate=sr))
    RUNS.append((title, audio_s, elapsed, allocated, peak))
    return wave, sr


ne_wav, ne_sr = speak(
    "Nepali / Maya preset",
    NE_TEXT,
    ref_wav=REF_WAV,
    ref_text=REF_TEXT,
    wav_path="swar-maya-nepali.wav",
)
_mix, _ = speak(
    "Mixed Nepali-English / Maya preset",
    MIX_TEXT,
    ref_wav=REF_WAV,
    ref_text=REF_TEXT,
    wav_path="swar-maya-mixed.wav",
)
_en, _ = speak(
    "English / Maya preset",
    EN_TEXT,
    ref_wav=REF_WAV,
    ref_text=REF_TEXT,
    wav_path="swar-maya-english.wav",
)
print("Nepali samples:", ne_wav.shape, "sr", ne_sr, "finite", bool(np.isfinite(ne_wav).all()))


## 5. Demo 2 — zero-shot clone from a synthetic reference

The reference is the **Nepali line just generated** — model output, CC BY-SA 4.0, with an exact transcript. It is **not** a recording of a real person. Do not point this at anyone's voice without their consent.

This is the same `task="TTS"` + prompt-wav pattern as F5: `ref_file` + `ref_text` + new `gen_text`. The official cloner would also enrol with Kriti and keep the best of several takes; one take is enough to show the API on a T4.


In [ ]:
CLONE_REF_WAV = "swar-maya-nepali.wav"
CLONE_REF_TEXT = NE_TEXT
CLONE_TEXT = "यो आवाज नमुनाबाट बनेको हो। अनुमति बिना कसैको आवाज नक्कल नगर्नुहोस्।"

info = sf.info(CLONE_REF_WAV)
print(
    f"clone reference {CLONE_REF_WAV}: {info.duration:.2f}s, {info.samplerate} Hz "
    "(synthetic Maya line from the previous cell)"
)
display(Audio(CLONE_REF_WAV))

_clone, _ = speak(
    "zero-shot clone / synthetic Maya ref",
    CLONE_TEXT,
    ref_wav=CLONE_REF_WAV,
    ref_text=CLONE_REF_TEXT,
    wav_path="swar-clone-synthetic.wav",
)
print("clone wrote swar-clone-synthetic.wav")


## 6. Peak VRAM and timing

RTF is wall-clock seconds per audio second; below 1.0 is faster than real time. Allocated memory after load is the float32 F5-TTS Base EMA plus Vocos. Peak is the NFE-16 decode step.


In [ ]:
header = f"{'clip':<40} {'audio_s':>8} {'wall_s':>8} {'rtf':>6} {'alloc_GB':>9} {'peak_GB':>8}"
print(header)
print("-" * 84)
for title, duration, elapsed, allocated, peak in RUNS:
    rtf = elapsed / duration if duration else 0.0
    print(f"{title:<40} {duration:8.2f} {elapsed:8.1f} {rtf:6.2f} {allocated:9.2f} {peak:8.2f}")
print()
print("GPU:", torch.cuda.get_device_name())
print("ckpt:", CKPT_NAME, "| nfe", NFE_STEP, "| cfg", CFG_STRENGTH, "| seed", SEED)
print("ema dtype:", next(tts.ema_model.parameters()).dtype)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"VRAM allocated now: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"VRAM peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("All demo waveforms were asserted finite at generate time.")


## 7. Caveats

- **Licence.** Weights are [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) (share-alike, commercial OK, attribution required). Code is MIT. Do not load `SWivid/F5-TTS` default weights here (CC BY-NC). Do not load `elitexp/Swar` (different model, CC BY-NC).
- **Consent.** Only clone a voice you have permission to use. Maya / Aarav are synthetic. The clone demo reuses a clip this notebook just synthesised. Disclose synthetic speech where that matters.
- **Install.** Official package name is `f5-tts` on PyPI, pin `1.1.22` (same as the authors' Colab). Do not install lookalike names. Do not replace Colab's torch. After install, restart the session once.
- **Float32 EMA.** `tts.ema_model.float()` is required. Float16 produces NaN or noise on IndicF5-derived checkpoints.
- **Memory.** This notebook downloads ~1.35 GB (`podvoices_ema.pt`) and skips `swar_v6_ema.pt` (~1.35 GB) and Kriti ONNX CTC (~0.49 GB). Keep `nfe_step=16` and short sentences on a 15 GB T4.
- **No-ref TTS.** There is none. Built-in Maya / Aarav presets are the no-upload path. F5 always needs `ref_file` + `ref_text`.
- **English.** Default `podvoices_ema.pt` can speak English from a Nepali prompt. `swar_v6_ema.pt` is stronger on English-only lines and weaker on pure Nepali; it is not loaded here.
- **Not the full cloner.** The Gradio app enrols with Kriti, scores several takes, and picks by pronunciation then voice match. This notebook is one-take `F5TTS.infer`.
- **Not Phonon, not Maya 2, not Drama 3.** Phonon 31M is 64.5 MB English CPU TTS. Maya 2 Native is 2.50B Indic streaming TTS. Drama 3-int4 needs an A100.
- **This file was not executed on a GPU in the pull request that added it.** Numbers above print at runtime on your Colab T4.


## 8. Cleanup

Release the F5-TTS module and VRAM when you are done. Re-run the load cell to start again.


In [ ]:
del tts
torch.cuda.empty_cache()
free_b, total_b = torch.cuda.mem_get_info()
print(
    f"after cleanup: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"free / total {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB"
)
